# W5D5 — Working with an LLM — Lab

**Week 5 · Day 5 · NLP Foundations** · Lab

All week you trained models from scratch, and on Thursday you built the transformer block yourself. A
large language model is a stack of those blocks — decoder blocks, with Thursday's causal mask —
pretrained on trillions of tokens to do exactly one thing: **predict the next token**.

Today you run one on your own CPU, with no API key: `Qwen2.5-0.5B-Instruct`, half a billion
parameters. You look at the real prompt behind a chat, read the model's next-token probabilities, write
the sampling that turns them into text, and then use it on the week's task — classifying the same 2,400
test reviews — **without training it at all**.

Then the leaderboard is complete: six kinds of model on the same data, with accuracy and cost side
by side. You decide which one you would ship.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ٥ — العمل مع نموذج لغة كبير

**الأسبوع الخامس · اليوم الخامس · أساسيات معالجة اللغة** · معمل

درّبت طوال الأسبوع نماذج من الصفر، وبنيت يوم الخميس كتلة المحوّل بنفسك. ونموذج اللغة الكبير (LLM) كومٌ من تلك
الكتل — كتل مُفكِّك، بقناع الخميس السببي — دُرِّب مسبقًا على تريليونات الرموز لشيء واحد بالضبط: **التنبّؤ بالرمز
التالي**.

وتُشغّل اليوم واحدًا على معالجك، بلا مفتاح API: `Qwen2.5-0.5B-Instruct`، نصف مليار معامل. تنظر إلى الموجِّه
(prompt) الحقيقي خلف المحادثة، وتقرأ احتمالات النموذج للرمز التالي، وتكتب أخذ العيّنات الذي يحوّلها إلى نص، ثم
تستعمله في مهمة الأسبوع — تصنيف مراجعات الاختبار نفسها، ٢٬٤٠٠ مراجعة — **دون أي تدريب**.

ثم تكتمل اللوحة: ستة أنواع من النماذج على البيانات نفسها، والدقّة والكلفة جنبًا إلى جنب. وتقرّر أيّها كنت لتنشره.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Read the real string a chat template produces, and count what it costs in tokens.
- Read an LLM's next-token distribution, and say what "predict the next token" means concretely.
- Implement temperature, top-k and top-p sampling, and reproduce greedy generation with your own loop.
- Classify text zero-shot and few-shot from next-token scores, and measure the cost.
- Get structured output from a model and measure how often it is actually valid.
- Recognise a hallucination, and choose between six models under a stated constraint.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تقرأ السلسلة الحقيقية التي يُنتجها قالب المحادثة، وتعدّ كلفتها بالرموز.
- أن تقرأ توزيع الرمز التالي في نموذج لغة كبير، وتقول ما معنى «التنبّؤ بالرمز التالي» بالملموس.
- أن تُنفّذ أخذ العيّنات بدرجة الحرارة وtop-k وtop-p، وتُعيد إنتاج التوليد الجشِع بحلقتك أنت.
- أن تُصنّف النص دون أمثلة وبأمثلة قليلة من درجات الرمز التالي، وتقيس الكلفة.
- أن تحصل على مخرجات منظّمة من النموذج وتقيس كم مرّة تكون صالحة فعلًا.
- أن تتعرّف على الهلوسة، وتختار بين ستة نماذج تحت قيدٍ مُعلَن.

</div>

## About the data

**The model:** `Qwen/Qwen2.5-0.5B-Instruct` (Apache-2.0), 494 million parameters, a context window of
32,768 tokens. **First-run download: about 1 GB**, then cached — start the setup cell early. It runs on
a laptop CPU at roughly half a second per review.

**The data:** `reviews_sentiment` on the week's split (`week5_split.json`, hash checked). Scoring all
2,400 test reviews would take about twenty minutes, so the lab scores a fixed sample of **200** of
them, and records that size in its leaderboard row. Every other row used all 2,400 — keep that in mind
when you compare.

**The probe:** one question about Olo Retail, the fictional company in week 7's `policy_docs`. Nothing
about it is in any model's training data.

<div dir="rtl" align="right">

## عن البيانات

**النموذج:** `Qwen/Qwen2.5-0.5B-Instruct` (رخصة Apache-2.0)، ٤٩٤ مليون معامل، ونافذة سياق من ٣٢٬٧٦٨ رمزًا.
**تنزيل أول تشغيل: نحو ١ غيغابايت**، ثم يُخزَّن — فابدأ خلية الإعداد مبكرًا. ويعمل على معالج حاسوب محمول بنحو نصف
ثانية للمراجعة.

**البيانات:** `reviews_sentiment` على تقسيم الأسبوع (`week5_split.json` مع التحقّق من البصمة). وتقييم مراجعات
الاختبار كلها، ٢٬٤٠٠ مراجعة، يستغرق نحو عشرين دقيقة، فيقيّم المعمل عيّنة ثابتة من **٢٠٠** منها، ويسجّل هذا الحجم
في صفّ اللوحة. وكل صف آخر استعمل الـ٢٬٤٠٠ كلها — فتذكّر ذلك حين تقارن.

**المِسبار:** سؤال واحد عن Olo Retail، الشركة المتخيَّلة في `policy_docs` في الأسبوع السابع. ولا شيء عنها في بيانات
تدريب أي نموذج.

</div>

## Setup

<div dir="rtl" align="right">

## الإعداد

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import get_dataset, get_dataset_dir, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, report

ensure("transformers", "matplotlib")
seed_everything(42)

import hashlib
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
NEGATION_PAIR = ["the service was good", "the service was not good"]
ORDER_PAIR = ["the food was good but the service was bad",
              "the service was good but the food was bad"]
torch.set_num_threads(4)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).eval()     # ~1 GB the first time
reviews = pd.read_parquet(get_dataset("reviews_sentiment"))
split = json.loads(load_artefact("week5_split.json").read_text(encoding="utf-8"))
check(hashlib.sha256(json.dumps({"train": split["train"], "test": split["test"]}).encode()).hexdigest()
      == split["sha256"], "week5_split.json must match its own hash", "يجب أن يطابق `week5_split.json` بصمته")
train, test = reviews.iloc[split["train"]], reviews.iloc[split["test"]]
print(f"train {len(train):,} | test {len(test):,} | split {split['sha256'][:12]}…")
print(versions())

In [ ]:
def load_board(days):
    """The leaderboard rows earlier days wrote (each day keeps its own file)."""
    rows = []
    for day in days:
        rows += json.loads(load_artefact(f"leaderboard_{day}.json").read_text(encoding="utf-8"))
    return rows


def show_board(rows):
    columns = ["day", "model", "n_train", "accuracy", "f1", "fit_s", "ms_per_1000", "size_mb",
               "negation_ok", "order_gap"]
    print(pd.DataFrame(rows)[columns].to_string(index=False))

## Section 1 — Warm-up: inside the model  (≈25 min)

Everything here works.

### Size, tokens and the context window

Count the parameters and read the context window from the config. Then tokenise two of the week's
strings. Like every model you will meet now, this one splits text into **subword** pieces: common words
stay whole, and a rarer word like `unhappiness` falls into parts. The `Ġ` marks a piece that starts with
a space.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: داخل النموذج (نحو ٢٥ دقيقة)

كل ما هنا يعمل.

### الحجم، والرموز، ونافذة السياق

عُدّ المعاملات واقرأ نافذة السياق من الإعدادات. ثم رمِّز نصّين من نصوص الأسبوع. ككل نموذج ستلقاه من الآن، يقسم هذا
النموذج النص إلى قطع **دون الكلمة** (subword): الكلمات الشائعة تبقى كاملة، والكلمة الأندر مثل `unhappiness` تتفكّك
أجزاءً. والعلامة `Ġ` تدلّ على قطعة تبدأ بمسافة.

</div>

In [ ]:
PARAMETERS = sum(p.numel() for p in model.parameters())
CONTEXT = model.config.max_position_embeddings
print(f"{PARAMETERS / 1e6:.0f} million parameters | {model.config.num_hidden_layers} decoder blocks | "
      f"context window {CONTEXT:,} tokens | vocabulary {len(tokenizer):,}")
for text in ["the service was not good", "unhappiness"]:
    print(f"{text!r:<28} -> {tokenizer.tokenize(text)}")

### The chat template

You send an instruct model a list of messages. The model never sees a list: the **chat template**
turns it into one string with special tokens marking who is speaking, and ends with the opening of the
assistant's turn, so the most likely next token is the start of a reply. Print that string and count
its tokens. The system prompt is paid for in tokens on every single request.

<div dir="rtl" align="right">

### قالب المحادثة

ترسل إلى نموذج التعليمات قائمة رسائل. ولا يرى النموذج قائمةً قطّ: **قالب المحادثة** يحوّلها إلى سلسلة واحدة برموز
خاصة تحدّد من يتكلّم، وتنتهي ببداية دور المساعد، فيكون أرجح رمز تالٍ بداية ردّ. اطبع تلك السلسلة وعُدّ رموزها.
فموجِّه النظام يُدفع ثمنه بالرموز في كل طلب على حدة.

</div>

In [ ]:
MESSAGES = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Is this review positive or negative? 'the service was not good'"},
]
TEMPLATE = tokenizer.apply_chat_template(MESSAGES, tokenize=False, add_generation_prompt=True)
print(TEMPLATE)
TEMPLATE_TOKENS = len(tokenizer(TEMPLATE).input_ids)
print(f"{TEMPLATE_TOKENS} tokens for a {len(MESSAGES[1]['content'].split())}-word question")

### The next token

One forward pass of the model returns a score — a **logit** — for every token in its vocabulary, at
every position. The last position's scores, through a softmax, are the probability of each possible
next token. Feed it `The capital of France is` and print the ten most likely continuations. That
distribution **is** the language model. Everything else today is a way of choosing from it.

<div dir="rtl" align="right">

### الرمز التالي

تُعيد تمريرة أمامية واحدة درجةً — **logit** — لكل رمز في المعجم، عند كل موضع. ودرجات الموضع الأخير، بعد softmax،
هي احتمال كل رمز تالٍ ممكن. أدخِل `The capital of France is` واطبع أرجح عشرة امتدادات. ذلك التوزيع **هو** نموذج
اللغة. وكل ما بعده اليوم طرائق للاختيار منه.

</div>

In [ ]:
def next_token_logits(text):
    """Logits for the token that comes after `text` (a 1-D tensor over the vocabulary)."""
    ids = tokenizer(text, return_tensors="pt").input_ids
    with torch.inference_mode():
        return model(ids).logits[0, -1]


probabilities = torch.softmax(next_token_logits("The capital of France is"), dim=-1)
top = torch.topk(probabilities, 10)
for p, i in zip(top.values, top.indices):
    print(f"  {p:.3f}  {tokenizer.decode(i)!r}")
TOP_TOKEN = tokenizer.decode(top.indices[0])

## Section 2 — Core: six tasks  (≈60 min)

1. Sampling by hand: temperature, top-k, top-p, and your own generation loop.
2. Decoding in practice: what temperature does to the text and to the distribution.
3. Zero-shot classification from next-token scores — the leaderboard row.
4. Few-shot: does showing examples help, and what does it cost?
5. Structured output: JSON, and how often it really is JSON.
6. The week's verdict — and one hallucination.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. أخذ العيّنات يدويًا: درجة الحرارة وtop-k وtop-p، وحلقة توليدك أنت.
٢. فكّ الترميز عمليًا: ما تفعله درجة الحرارة بالنص وبالتوزيع.
٣. التصنيف دون أمثلة من درجات الرمز التالي — صفّ اللوحة.
٤. الأمثلة القليلة: هل يساعد عرض الأمثلة، وما كلفته؟
٥. المخرجات المنظّمة: JSON، وكم مرّة يكون JSON حقًا.
٦. حكم الأسبوع — وهلوسة واحدة.

</div>

### Task 2.1 — sampling, by hand

Write `sample(logits, temperature, top_k, top_p)`:

- **temperature** divides the logits before the softmax. Below 1 it sharpens the distribution, above 1
  it flattens it. At 0 it becomes the **argmax** — greedy decoding — handle that case separately.
- **top-k** keeps only the `k` most likely tokens.
- **top-p** (nucleus) keeps the **smallest** set of most-likely tokens whose probabilities add up to at
  least `p`.

Then put it in a loop: score the text so far, pick a token, append it, repeat. With temperature 0 your
loop must produce **exactly** what `model.generate(do_sample=False)` produces — that is what generation
is. Thursday's causal mask is what makes this valid: the model was trained never to look ahead.

One catch you will meet in practice: this checkpoint ships default generation settings, including a
**repetition penalty** that `generate()` applies even with sampling off. The comparison switches it
off. Always check what a library's "defaults" quietly add to decoding.

<div dir="rtl" align="right">

### المهمة ٢٫١ — أخذ العيّنات يدويًا

اكتب `sample(logits, temperature, top_k, top_p)`:

- **درجة الحرارة** تقسم الـlogits قبل softmax. ما دون ١ يُحدّد التوزيع، وما فوقه يُسطّحه. وعند ٠ يصير **الأرجح**
  (argmax) — فكّ الترميز الجشِع — فعالج هذه الحالة على حدة.
- **top-k** يُبقي أرجح `k` رموز فقط.
- **top-p** (النواة) يُبقي **أصغر** مجموعة من الرموز الأرجح يبلغ مجموع احتمالاتها `p` على الأقل.

ثم ضعها في حلقة: قيّم النص حتى الآن، واختر رمزًا، وألحِقه، وكرّر. وبدرجة حرارة ٠ يجب أن تُنتج حلقتك **بالضبط** ما
يُنتجه `model.generate(do_sample=False)` — فهذا هو التوليد. وقناع الخميس السببي هو ما يجعل ذلك صحيحًا: فقد دُرِّب
النموذج على ألّا ينظر إلى الأمام قطّ.

ومطبّ ستلقاه عمليًا: يأتي هذا النموذج بإعدادات توليد افتراضية، منها **عقوبة تكرار** يطبّقها `generate()` حتى مع
إطفاء أخذ العيّنات. والمقارنة تُطفئها. فتحقّق دائمًا مما تضيفه «افتراضيات» المكتبة إلى فكّ الترميز خِفيةً.

</div>

In [ ]:
def sample(logits, temperature=1.0, top_k=None, top_p=None, generator=None):
    """Pick one token id from next-token logits."""
    # TODO: Temperature 0 -> return the argmax. Otherwise divide by the temperature, softmax, keep the top_k tokens if given, keep the smallest set reaching top_p if given, renormalise, and draw one id.
    # مهمة: درجة حرارة ٠ ← أعِد الأرجح. وإلا فاقسم على درجة الحرارة، وطبّق softmax، وأبقِ أرجح `top_k` رموز إن أُعطي، وأبقِ أصغر مجموعة تبلغ `top_p` إن أُعطي، وأعِد التطبيع، واسحب رقمًا واحدًا.

In [ ]:
def generate_by_hand(prompt, max_new_tokens=20, **sampling):
    """Score the text so far, sample one token, append it, repeat."""
    ids = tokenizer(prompt, return_tensors="pt").input_ids
    for _ in range(max_new_tokens):
        # TODO: Get the last position's logits for `ids`, sample a token, stop on the end-of-text token, otherwise append it to `ids`.
        # مهمة: خُذ logits الموضع الأخير لـ`ids`، واسحب رمزًا، وتوقّف عند رمز نهاية النص، وإلا فألحِقه بـ`ids`.
    return ids


PROMPT = "The capital of France is"
mine = generate_by_hand(PROMPT, temperature=0)
with torch.inference_mode():
    # The checkpoint ships a generation config with repetition_penalty=1.1, which generate() applies
    # even when do_sample=False — switch it off to compare plain greedy decoding. use_cache=False makes
    # generate() recompute every step exactly as the loop does.
    theirs = model.generate(tokenizer(PROMPT, return_tensors="pt").input_ids, max_new_tokens=20,
                            do_sample=False, repetition_penalty=1.0, use_cache=False,
                            pad_token_id=tokenizer.eos_token_id)
n = min(mine.shape[1], theirs.shape[1])            # generate() may also stop on a second end-of-text token
GREEDY_MATCHES = torch.equal(mine[0, :n], theirs[0, :n])
print("my loop :", tokenizer.decode(mine[0]))
print("generate:", tokenizer.decode(theirs[0]))
print(f"identical: {GREEDY_MATCHES}")

### Task 2.2 — decoding in practice

Ask the chat model one open question at three temperatures — 0, 0.7 and 1.5 — three times each, with
`top_p = 0.95`. At 0 the three answers are identical, every time. At 0.7 they differ and stay sensible.
At 1.5 they drift.

Then measure why. The **entropy** of the next-token distribution — `−Σ p·log p` — is low when the
model is sure and high when it is spread out. Compute it for one prompt at each temperature: it rises
monotonically with the temperature, which is all temperature is.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — فكّ الترميز عمليًا

اسأل نموذج المحادثة سؤالًا مفتوحًا واحدًا بثلاث درجات حرارة — ٠ و٠٫٧ و١٫٥ — ثلاث مرّات لكلٍّ منها، مع
`top_p = 0.95`. عند ٠ تتطابق الأجوبة الثلاثة، في كل مرّة. وعند ٠٫٧ تختلف وتبقى معقولة. وعند ١٫٥ تنجرف.

ثم قِس السبب. **إنتروبيا** توزيع الرمز التالي — `−Σ p·log p` — منخفضة حين يكون النموذج واثقًا ومرتفعة حين يتوزّع.
احسبها لموجِّه واحد عند كل درجة حرارة: فهي ترتفع باطّراد مع درجة الحرارة، وهذا كل ما تفعله درجة الحرارة.

</div>

In [ ]:
QUESTION = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Describe a rainy day in one sentence."}], tokenize=False, add_generation_prompt=True)
generator = torch.Generator().manual_seed(SEED)
for temperature in [0, 0.7, 1.5]:
    print(f"temperature {temperature}:")
    for _ in range(3):
        ids = generate_by_hand(QUESTION, max_new_tokens=25, temperature=temperature, top_p=0.95, generator=generator)
        reply = tokenizer.decode(ids[0], skip_special_tokens=True).split("assistant\n")[-1]
        print(f"   {reply.strip()!r}")

In [ ]:
# TODO: For temperatures 0.25, 0.5, 1, 1.5 and 2, compute the entropy of softmax(logits / T) for QUESTION.
# مهمة: لدرجات الحرارة ٠٫٢٥ و٠٫٥ و١ و١٫٥ و٢، احسب إنتروبيا `softmax(logits / T)` لـ`QUESTION`.
print(pd.Series(ENTROPY, name="entropy (nats)").round(3).to_string())

### Task 2.3 — zero-shot classification

The model has never been trained on these reviews. Ask it anyway: a system prompt saying "answer with
one word: positive or negative", the review as the user message, and the chat template's opening of
the assistant's turn. Then **don't generate at all**. Read the next-token logits for `positive` and
`negative` and compare them. One forward pass per review, and the answer can only be one of the two
labels.

Score the 200-review sample, time it, and add the leaderboard row — `n_train` is **0**. Run both
sentence pairs too.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — التصنيف دون أمثلة

لم يُدرَّب النموذج على هذه المراجعات قطّ. فاسأله مع ذلك: موجِّه نظام يقول «أجِب بكلمة واحدة: positive أو
negative»، والمراجعة رسالةَ المستخدم، وبداية دور المساعد من قالب المحادثة. ثم **لا تولِّد شيئًا**. اقرأ logits
الرمز التالي لـ`positive` و`negative` وقارنهما. تمريرة أمامية واحدة لكل مراجعة، ولا يمكن أن يكون الجواب إلا إحدى
التسميتين.

قيّم عيّنة المراجعات المئتين، ووقّتها، وأضِف صفّ اللوحة — و`n_train` فيه **٠**. وشغّل زوجَي الجمل أيضًا.

</div>

In [ ]:
SYSTEM = "Classify the sentiment of product reviews. Answer with one word: positive or negative."
POSITIVE_ID, NEGATIVE_ID = tokenizer.encode("positive")[0], tokenizer.encode("negative")[0]
sample_200 = test.sample(200, random_state=SEED)


def classification_prompt(review, examples=()):
    messages = [{"role": "system", "content": SYSTEM}]
    for text, label in examples:
        messages += [{"role": "user", "content": f"Review: {text}"}, {"role": "assistant", "content": label}]
    messages.append({"role": "user", "content": f"Review: {review}"})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def positive_margin(review, examples=()):
    """logit('positive') - logit('negative') for the assistant's first token. Above 0 means positive."""
    # TODO: Build the prompt, get the next-token logits, and return the difference of the two label logits.
    # مهمة: ابنِ الموجِّه، وخُذ logits الرمز التالي، وأعِد الفرق بين logit التسميتين.

In [ ]:
start = time.perf_counter()
MARGINS = np.array([positive_margin(r) for r in sample_200.review_text])     # about 100 s on a laptop
LLM_MS_PER_1000 = (time.perf_counter() - start) / len(sample_200) * 1e6
zero_shot = (MARGINS > 0).astype(int)
ZERO_SHOT_ACCURACY = float((zero_shot == sample_200.sentiment.to_numpy()).mean())
print(f"zero-shot accuracy {ZERO_SHOT_ACCURACY:.3f} on {len(sample_200)} test reviews | "
      f"{LLM_MS_PER_1000 / 1e6:.2f} s per review | predicted positive {zero_shot.mean():.0%}")

In [ ]:
from sklearn.metrics import f1_score

pair_margins = [positive_margin(s) for s in NEGATION_PAIR + ORDER_PAIR]
p_pairs = 1 / (1 + np.exp(-np.array(pair_margins)))          # the two-label softmax
for sentence, p in zip(NEGATION_PAIR + ORDER_PAIR, p_pairs):
    print(f"  p(positive) = {p:.4f}   {sentence}")

BOARD_D5 = [{
    "day": "D5", "model": "Qwen2.5-0.5B-Instruct, zero-shot", "n_train": 0,
    "accuracy": round(ZERO_SHOT_ACCURACY, 4), "f1": round(f1_score(sample_200.sentiment, zero_shot), 4),
    "fit_s": 0.0, "ms_per_1000": round(LLM_MS_PER_1000, 1), "size_mb": round(PARAMETERS * 4 / 1e6, 1),
    "negation_ok": bool(p_pairs[0] > 0.5 > p_pairs[1]), "order_gap": round(float(abs(p_pairs[2] - p_pairs[3])), 4),
    "n_test": len(sample_200),
}]

### Task 2.4 — few-shot

Put four labelled training reviews into the prompt as earlier turns of the conversation — the model
reads them as examples of the task — and score the **first 50** reviews of the sample again, both ways.
Count the prompt tokens.

Few-shot prompting is the standard way to teach an LLM a task without training it, and on weaker
models it often helps a lot. On this one, which already does the task well zero-shot, it may not move
the accuracy at all, while every request now carries several hundred extra tokens. Measure both before
you decide it is worth it.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — الأمثلة القليلة

ضع أربع مراجعات تدريب مُسمّاة في الموجِّه أدوارًا سابقة في المحادثة — فيقرؤها النموذج أمثلةً على المهمة — وقيّم
**أول ٥٠** مراجعة من العيّنة مرّة أخرى بالطريقتين. وعُدّ رموز الموجِّه.

الموجِّه بأمثلة قليلة (few-shot) هو الطريقة المعتادة لتعليم نموذج لغة كبير مهمةً دون تدريبه، وعلى النماذج الأضعف
كثيرًا ما يساعد كثيرًا. أمّا على هذا النموذج، الذي يؤدّي المهمة جيدًا أصلًا دون أمثلة، فقد لا يحرّك الدقّة إطلاقًا،
بينما يحمل كل طلب الآن بضع مئات من الرموز الإضافية. قِس الأمرين قبل أن تقرّر أنه يستحق.

</div>

In [ ]:
EXAMPLES = [(train.review_text.iloc[i][:200], "positive" if train.sentiment.iloc[i] else "negative")
            for i in range(4)]
first_50 = sample_200.iloc[:50]

# TODO: Time the few-shot margins on first_50, compare both accuracies on those 50, and count the prompt tokens.
# مهمة: وقّت هوامش الأمثلة القليلة على `first_50`، وقارن الدقّتين على تلك الخمسين، وعُدّ رموز الموجِّه.
print(f"on the same 50 reviews: zero-shot {ZERO_ON_50:.3f} | few-shot {FEW_SHOT_ACCURACY:.3f}")
print(f"prompt tokens {PROMPT_TOKENS} | few-shot costs {FEW_SHOT_S_PER_REVIEW:.2f} s per review")

### Task 2.5 — structured output

A program downstream wants JSON, not prose. Ask for it: `{"sentiment": "positive" or "negative",
"reason": "..."}`, on 10 reviews, generating up to 60 tokens greedily. Then try to `json.loads` each
answer and check that `sentiment` is one of the two allowed values.

Some answers will come back wrapped in a markdown code fence, some cut off when the 60 tokens run out,
some valid. The parse-success rate is the number to report. Asking nicely is not a guarantee — production
systems validate every response and retry, or use constrained decoding that cannot produce invalid
output.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — المخرجات المنظّمة

برنامجٌ لاحق يريد JSON لا نثرًا. فاطلبه: `{"sentiment": "positive" أو "negative", "reason": "..."}`، على ١٠
مراجعات، بتوليد جشِع حتى ٦٠ رمزًا. ثم حاول `json.loads` على كل جواب وتحقّق أن `sentiment` إحدى القيمتين المسموحتين.

ستعود بعض الأجوبة ملفوفةً في سياج شيفرة markdown، وبعضها مقطوعًا حين تنفد الرموز الستّون، وبعضها صالحًا. ونسبة
نجاح التحليل هي الرقم الذي تعرضه. فالطلب بلطف ليس ضمانًا — والأنظمة في الإنتاج تتحقّق من كل ردّ وتُعيد المحاولة،
أو تستعمل فكّ ترميز مقيّدًا لا يستطيع إنتاج مخرجات غير صالحة.

</div>

In [ ]:
def ask(question, max_new_tokens=60):
    """Greedy chat reply to one user message."""
    ids = tokenizer(tokenizer.apply_chat_template([{"role": "user", "content": question}], tokenize=False,
                                                  add_generation_prompt=True), return_tensors="pt").input_ids
    with torch.inference_mode():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()


# TODO: For 10 reviews, ask for the JSON, try json.loads (strip a ``` fence first), and count the answers that parse AND have sentiment in {"positive", "negative"}.
# مهمة: لعشر مراجعات، اطلب JSON، وجرّب `json.loads` (بعد نزع سياج ``` أولًا)، وعُدّ الأجوبة التي تُحلَّل وفيها `sentiment` من `{"positive", "negative"}`.
print(f"valid JSON with an allowed sentiment: {VALID} of 10\n")
for answer in ANSWERS[:3]:
    print(" ", answer[:120].replace("\n", " "))

### Task 2.6 — the week's verdict, and one hallucination

**The hallucination.** Ask the model how many days a customer has to return an item at **Olo Retail**.
Olo Retail does not exist: it is the fictional company in week 7's documents, whose real policy is **14
days**. The model has never seen it — and answers anyway, fluently and specifically. That is a
hallucination: not a malfunction, but the model doing exactly its one job — producing a likely next
token — about something it has no information on. Week 7 fixes it by putting the document in the prompt.

**The verdict.** Print the whole week's leaderboard. Then decide, in a short paragraph: you must serve
this classifier on a CPU with **at most 50 ms per request**, and it will see some Arabic reviews. Which
model do you ship, and what would change your mind? There is no single right answer; the reasoning,
with numbers from the table, is what is graded.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — حكم الأسبوع، وهلوسة واحدة

**الهلوسة.** اسأل النموذج كم يومًا لدى الزبون ليُعيد سلعة في **Olo Retail**. وOlo Retail غير موجودة: إنها الشركة
المتخيَّلة في وثائق الأسبوع السابع، وسياستها الحقيقية **١٤ يومًا**. ولم يرَها النموذج قطّ — ويُجيب مع ذلك، بطلاقة
وتحديد. وهذه هلوسة: لا عطلٌ، بل النموذج يؤدّي عمله الوحيد بالضبط — إنتاج رمز تالٍ مرجّح — عن شيء لا معلومات له
عنه. ويُصلحها الأسبوع السابع بوضع الوثيقة في الموجِّه.

**الحكم.** اطبع لوحة الأسبوع كلها. ثم قرّر في فقرة قصيرة: عليك أن تُقدّم هذا المُصنّف على معالج بما **لا يتجاوز ٥٠
مللي ثانية للطلب**، وسيرى بعض المراجعات العربية. أيّ نموذج تنشر، وما الذي يغيّر رأيك؟ لا جواب واحدًا صحيحًا؛
والمُقيَّم هو الاستدلال، بأرقام من الجدول.

</div>

In [ ]:
policy = (get_dataset_dir("policy_docs") / "docs" / "POL-101.md").read_text(encoding="utf-8")
TRUTH = next(line for line in policy.splitlines() if "within 14 days" in line)
HALLUCINATION = ask("How many days does a customer have to return an item at Olo Retail? Answer in one sentence.")
print(f"the model : {HALLUCINATION}")
print(f"the policy: …{TRUTH[TRUTH.index('A refund'):][:80]}…")

In [ ]:
FULL_BOARD = load_board(["D1", "D2", "D3", "D4"]) + BOARD_D5
show_board(FULL_BOARD)

VERDICT = (
    "Under 50 ms per request on a CPU, TF-IDF, the GRU and the TCN all qualify easily; the LLM is the most "
    f"accurate model on the board without any training, but at about {LLM_MS_PER_1000 / 1000:.0f} ms per review "
    f"it is {LLM_MS_PER_1000 / 1000 / 50:.0f} times over budget. I would ship TF-IDF + logistic regression: it is the most accurate model that fits, "
    "it is the smallest, and it retrains in under a second. What would change my mind: a GPU budget, or a "
    "requirement to handle the Arabic reviews well — then the LLM (or a multilingual fine-tuned encoder) is worth "
    "its cost."
)
print(f"\nverdict: {VERDICT}")

## Section 3 — Stretch  (≈30 min)

**(a) What a prompt costs.** Time one forward pass for prompts of 64, 256, 1,024 and 2,048 tokens, and
plot the time against the length. Attention compares every token with every other one, so the cost
grows faster than the length. That is why long contexts cost money and why week 7 retrieves only the
relevant chunks.

**(b) A big model, through an API.** If you have an OpenRouter key in `.env` (see
`docs/Data_Guide.md`), score the first 20 sample reviews with a large hosted model through
`aiep.llm.chat`, which caches every response on disk. Compare its accuracy and latency with the
0.5-billion-parameter model on your CPU. Without a key, the cell says so and skips.

<div dir="rtl" align="right">

## القسم الثالث — إضافي (نحو ٣٠ دقيقة)

**(أ) كلفة الموجِّه.** وقّت تمريرة أمامية واحدة لموجِّهات من ٦٤ و٢٥٦ و١٬٠٢٤ و٢٬٠٤٨ رمزًا، وارسم الزمن مقابل
الطول. فالانتباه يقارن كل رمز بكل رمز آخر، فتنمو الكلفة أسرع من الطول. ولهذا تكلّف السياقات الطويلة مالًا، ولهذا
لا يسترجع الأسبوع السابع إلا المقاطع ذات الصلة.

**(ب) نموذج كبير عبر API.** إن كان لديك مفتاح OpenRouter في `.env` (انظر `docs/Data_Guide.md`)، فقيّم أول ٢٠
مراجعة من العيّنة بنموذج كبير مُستضاف عبر `aiep.llm.chat`، الذي يخزّن كل ردّ على القرص. وقارن دقّته وزمنه
بالنموذج ذي نصف المليار معامل على معالجك. وبلا مفتاح تقول الخلية ذلك وتتخطّى.

</div>

In [ ]:
LATENCY = {}
filler = " ".join(train.review_text.iloc[:200])
for n_tokens in [64, 256, 1024, 2048]:
    ids = tokenizer(filler, return_tensors="pt").input_ids[:, :n_tokens]
    with torch.inference_mode():
        model(ids)                                             # warm-up
        start = time.perf_counter()
        model(ids)
    LATENCY[n_tokens] = time.perf_counter() - start
pd.Series(LATENCY).plot(marker="o", title="one forward pass against prompt length")
plt.xlabel("tokens")
plt.ylabel("seconds")
plt.show()
print({n: round(s, 2) for n, s in LATENCY.items()})

In [ ]:
from aiep.llm import chat

try:
    start = time.perf_counter()
    answers = [chat(f"Review: {r}", system=SYSTEM, max_tokens=3)["text"] for r in sample_200.review_text.iloc[:20]]
    api_accuracy = np.mean([a.strip().lower().startswith("pos") == bool(y)
                            for a, y in zip(answers, sample_200.sentiment.iloc[:20])])
    print(f"hosted model: accuracy {api_accuracy:.2f} on 20 reviews, "
          f"{(time.perf_counter() - start) / 20:.2f} s per review (cached calls are instant)")
except Exception as error:                                   # no key and no cached answers
    print(f"skipped — {type(error).__name__}: {str(error).strip().splitlines()[0] if str(error).strip() else 'no key and no cached answers'}")

## Save your artefacts

- `leaderboard_D5.json` — today's row.
- `week5_verdict.md` — the full leaderboard and your verdict. A one-page summary of the week.

<div dir="rtl" align="right">

## احفظ آثارك

- `leaderboard_D5.json` — صفّ اليوم.
- `week5_verdict.md` — اللوحة كاملة وحكمك. ملخّص الأسبوع في صفحة واحدة.

</div>

In [ ]:
(ARTEFACT_DIR / "leaderboard_D5.json").write_text(json.dumps(BOARD_D5, indent=2), encoding="utf-8")
columns = ["day", "model", "n_train", "accuracy", "f1", "fit_s", "ms_per_1000", "size_mb", "negation_ok", "order_gap"]
table = "\n".join(["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
                  + ["| " + " | ".join(str(row.get(c, "")) for c in columns) + " |" for row in FULL_BOARD])
(ARTEFACT_DIR / "week5_verdict.md").write_text(
    "# Week 5 — six models, one split\n\n" + table
    + "\n\nThe D5 row is scored on 200 of the 2,400 test reviews; every other row on all 2,400.\n\n"
    + f"## Verdict\n\n{VERDICT}\n\n## One hallucination\n\nAsked about Olo Retail's return window "
    + f"(14 days), the model answered: \"{HALLUCINATION}\"\n", encoding="utf-8")
print((ARTEFACT_DIR / "week5_verdict.md").read_text(encoding="utf-8")[:900])

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.5, -1.0])
draws = [sample(logits, temperature=1.0, top_k=2, generator=torch.Generator().manual_seed(i)) for i in range(200)]
check(sample(logits, temperature=0) == 0 and set(draws) <= {0, 1} and len(set(draws)) == 2,
      "temperature 0 must return the argmax, and top_k=2 must only ever draw the two most likely tokens",
      "يجب أن تُعيد درجة الحرارة ٠ الأرجح، وألّا يسحب `top_k=2` إلا أرجح رمزين")
# probabilities are about 0.61, 0.22, 0.14, 0.03: the smallest set reaching 0.8 is the first two
nucleus_draws = {sample(logits, temperature=1.0, top_p=0.8, generator=torch.Generator().manual_seed(i)) for i in range(200)}
check(nucleus_draws == {0, 1},
      f"top_p=0.8 must keep exactly the smallest set reaching 0.8 — here tokens 0 and 1 — got {sorted(nucleus_draws)}",
      f"يجب أن يُبقي `top_p=0.8` أصغر مجموعة تبلغ ٠٫٨ بالضبط — هنا الرمزان ٠ و١ — والناتج {sorted(nucleus_draws)}")
check(GREEDY_MATCHES and "Paris" in TOP_TOKEN,
      "your greedy loop must reproduce model.generate exactly, and ' Paris' must be the top next token",
      "يجب أن تُعيد حلقتك الجشِعة إنتاج `model.generate` بالضبط، وأن يكون ' Paris' أرجح رمز تالٍ")
check(all(a < b for a, b in zip(list(ENTROPY.values()), list(ENTROPY.values())[1:])),
      f"the next-token entropy must rise with the temperature — got {ENTROPY}",
      f"يجب أن ترتفع إنتروبيا الرمز التالي مع درجة الحرارة — والناتج {ENTROPY}")
check(ZERO_SHOT_ACCURACY > 0.8 and BOARD_D5[0]["n_train"] == 0,
      f"the untrained model must classify the sample zero-shot above 0.8 — got {ZERO_SHOT_ACCURACY:.3f}",
      f"يجب أن يُصنّف النموذج غير المُدرَّب العيّنة دون أمثلة فوق ٠٫٨ — والناتج {ZERO_SHOT_ACCURACY:.3f}")
check(PROMPT_TOKENS["few-shot"] > 2 * PROMPT_TOKENS["zero-shot"] and 0 <= VALID <= 10,
      f"the few-shot prompt must cost far more tokens than the zero-shot one — got {PROMPT_TOKENS}",
      f"يجب أن يكلّف موجِّه الأمثلة القليلة رموزًا أكثر بكثير من موجِّه بلا أمثلة — والناتج {PROMPT_TOKENS}")
check(len(FULL_BOARD) >= 5 and len(VERDICT.split()) > 30,
      "the full leaderboard must hold every day's rows, and the verdict must be a real paragraph",
      "يجب أن تحمل اللوحة الكاملة صفوف كل الأيام، وأن يكون الحكم فقرةً حقيقية")
report()

## What's next

The week is done. You turned text into vectors two ways, read it in order with an RNN, through a window
with a TCN, all at once with attention — and then used a model built from those same blocks, trained by
someone else on more text than you will ever see, to beat all of them without training anything.

**Week 6** takes the transformer to images: a Vision Transformer cuts a picture into patches and treats
them as tokens, with the same attention you wrote on Thursday. **Week 7** comes back to today's
hallucination: retrieve the right document, put it in the prompt, and make the model cite it.

<div dir="rtl" align="right">

## ما التالي

انتهى الأسبوع. حوّلت النص إلى متّجهات بطريقتين، وقرأته بالترتيب بـRNN، وعبر نافذة بـTCN، ودفعةً واحدة بالانتباه —
ثم استعملت نموذجًا مبنيًا من تلك الكتل نفسها، دُرِّب على يد غيرك على نصٍّ أكثر مما ستراه يومًا، لتتفوّق عليها كلها
دون أن تُدرّب شيئًا.

**والأسبوع السادس** يأخذ المحوّل إلى الصور: يقطّع محوّل الرؤية (Vision Transformer) الصورة رُقعًا ويعاملها رموزًا،
بالانتباه نفسه الذي كتبته يوم الخميس. **والأسبوع السابع** يعود إلى هلوسة اليوم: استرجِع الوثيقة الصحيحة، وضعها في
الموجِّه، واجعل النموذج يستشهد بها.

</div>